In [1]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install everything
!pip install -q fastapi uvicorn nest_asyncio pyngrok diffusers transformers accelerate safetensors torch torchvision python-multipart

# Ask for your key once (you paste it once here)
DEXGEN_API_KEY = input("Paste your DEXGEN_API_KEY from your Mac terminal and press Enter: ").strip()

# Prepare output folder
import os
OUTPUT_DIR = "/content/drive/MyDrive/DexGen/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load model
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)
pipe.to("cuda")
pipe.enable_attention_slicing()

# Start API
import nest_asyncio, uvicorn, threading, time
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel

app = FastAPI()

class PromptRequest(BaseModel):
    prompt: str
    steps: int = 30
    width: int = 512
    height: int = 512

@app.post("/generate")
def generate(data: PromptRequest, x_api_key: str = Header(...)):
    if x_api_key != DEXGEN_API_KEY:
        raise HTTPException(status_code=401, detail="Unauthorized")

    image = pipe(
        data.prompt,
        num_inference_steps=data.steps,
        width=data.width,
        height=data.height
    ).images[0]

    filename = f"{OUTPUT_DIR}/output_{int(time.time())}.png"
    image.save(filename)

    return JSONResponse({"saved_to": filename})

nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000)).start()

from pyngrok import ngrok
public_url = ngrok.connect(8000)
print("DexGen URL:", public_url)

Mounted at /content/drive
Paste your DEXGEN_API_KEY from your Mac terminal and press Enter: hovEjEYbFSQJvA4gtr_2fAmtvnZCDAGIo_en8xjEZMzlnWeBO9qkEK3WsoQxCCAK


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


INFO:     Started server process [475]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


ERROR:pyngrok.process.ngrok:t=2026-02-28T08:06:57+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-02-28T08:06:57+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-02-28T08:06:57+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [2]:
!pip install -q pyngrok
from pyngrok import ngrok

ngrok.set_auth_token("3AI3EfBKCyyV1TzdYtSsV9XCpHC_3fzrgJCMUbwTrhwcMxv74")

public_url = ngrok.connect(8000)
print("DexGen URL:", public_url)

DexGen URL: NgrokTunnel: "https://acrologic-spikily-bernadette.ngrok-free.dev" -> "http://localhost:8000"
